## Prerequisites

This notebook demonstrates how to invoke a **deployed real-time inference endpoint** on Azure for the ICD-10-CM Clinical Terminology Mapper model. Follow the steps below in order before running this notebook.

### Deployment Sequence

1. **Acquire the Azure Application Offer** — Subscribe to the *ICD-10-CM Clinical Terminology Mapper* offer in the [Azure Marketplace](https://azuremarketplace.microsoft.com) to obtain the Solution Template.
2. **Launch guided deployment** — From the Marketplace listing, start the guided deployment flow for the solution.
3. **Supply configuration** — Complete the guided UX forms (defined by the offer's `createUiDefinition`) with the required parameters.
4. **Deploy resources** — Confirm deployment; Azure provisions the resources defined in the offer's `mainTemplate.json`.
5. **Wait for deployment to succeed** — Verify the deployment reaches a *Succeeded* state before proceeding. Note the endpoint name from the deployment output.
6. **Run this notebook** — Execute the cells below against the real-time inference endpoint provisioned by the deployment.

### Environment

- **Notebook runtime**: This notebook is intended to run in **Azure Machine Learning workspace notebooks** (e.g. via an Azure ML Studio compute instance or Studio notebooks).
- **Jupyter kernel**: Use a kernel with the `azure.ai.ml` Python package installed, for example **Python 3.10 — SDK v2** (`python310-sdkv2`).
- **Authentication**: The kernel must be authenticated to Azure so that Azure ML SDK calls succeed (e.g. `DefaultAzureCredential()`). The authenticated identity must be granted Azure RBAC permissions appropriate for reading the workspace and online endpoint configuration, and for retrieving endpoint credentials. Consult your organization's access-policy guidance and the [Azure ML endpoint credential documentation](https://learn.microsoft.com/azure/machine-learning/how-to-authenticate-online-endpoint) for details.
- **Network access**: The notebook's compute instance must be able to reach the endpoint's `scoring_uri` (no outbound firewall or VNET restrictions that block it).


## ICD-10-CM Clinical Terminology Mapper

- **Model**: [icd10cm_vdb_resolver](https://nlp.johnsnowlabs.com/2024/08/22/icd10cm_resolver_pipeline_en.html)

- **Model Description**: This pretrained pipeline extracts clinical entities from clinical text and maps them to their corresponding ICD-10-CM codes.

In [2]:
ENDPOINT_NAME = "<CUSTOMER_TO_PUT_DEPLOYED_ENDPOINT_NAME>"

### Input Format

To use the model, you need to provide input in one of the following supported formats:

#### JSON Format

Provide input as JSON. We support two variations within this format:

1. **Array of Text Documents**:
   Use an array containing multiple text documents. Each element represents a separate text document.

   ```json
   {
       "text": [
           "Text document 1",
           "Text document 2",
           ...
       ]
   }
   ```

2. **Single Text Document**:
   Provide a single text document as a string.


   ```json
   {
       "text": "Single text document"
   }
   ```

#### JSON Lines (JSONL) Format

Provide input in JSON Lines format, where each line is a JSON object representing a text document.

```
{"text": "Text document 1"}
{"text": "Text document 2"}
```

### Authentication and Endpoint Lookup

The cell below uses `DefaultAzureCredential()` to authenticate to Azure, then retrieves the endpoint's scoring URI and primary access key from the workspace configuration. If you see authentication errors, verify that you are logged into Azure Cloud Shell / your compute instance and that your identity has sufficient role assignments on the workspace.


In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import pandas as pd
import requests
import json

# Set display options
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


ml_client = MLClient.from_config(DefaultAzureCredential())
endpoint_url = ml_client.online_endpoints.get(ENDPOINT_NAME).scoring_uri
key = ml_client.online_endpoints.get_keys(ENDPOINT_NAME).primary_key


def invoke_realtime_endpoint(record, content_type="application/json", accept="application/json"):
    response = requests.post(
        endpoint_url,
        headers={
            "Authorization": f"Bearer {key}",
            "Content-Type": content_type,
        },
        data=json.dumps(record) if content_type == "application/json" else record,

    )

    response_body = response.content.decode("utf-8")

    if accept == "application/json":
        return json.loads(response_body)
    elif accept == "application/jsonlines":
        return response_body
    else:
        raise ValueError(f"Unsupported accept type: {accept}")

### Sample Data (Synthetic — for demonstration only)

The following cells define sample clinical text documents used in the examples below.

In [4]:
docs = [
    "An 86-year-old female with persistent abdominal pain, nausea and projectile vomiting, during evaluation in the emergency room, was found to have a high amylase, as well as lipase count and she is being admitted for management of unspecified gastrointestinal hemorrhage.",
    "Complaints of unspecified upper abdominal pain and swelling in a 32-year-old woman led to the evaluation of possible disease of intestine. She had a history of mixed irritable bowel syndrome and congenital lactase deficiency. Several diagnostic tests were performed. Blood tests showed abnormal results of blood chemistry and colonoscopy was performed.",
]


sample_text = "A 28-year-old female with a history of gestational diabetes mellitus diagnosed eight years prior to presentation and subsequent type two diabetes mellitus, associated with an acute hepatitis, and obesity with a body mass index (BMI) of 33.5 kg/m2, presented with a one-week history of polyuria, polydipsia, poor appetite, and vomiting. Two weeks prior to presentation, she was treated with a five-day course of amoxicillin for a respiratory tract infection. She had been on dapagliflozin for six months at the time of presentation. Physical examination on presentation was significant for dry oral mucosa; significantly, her abdominal examination was benign with no tenderness or guarding. Pertinent laboratory findings on admission were: serum glucose 111 mg/dl, bicarbonate 18 mmol/l, anion gap 20, creatinine 0.4 mg/dL, triglycerides 508 mg/dL, total cholesterol 122 mg/dL, glycated hemoglobin (HbA1c) 10%, and venous pH 7.27. Serum lipase was normal at 43 U/L. Serum acetone levels could not be assessed as blood samples kept hemolyzing due to significant lipemia. The patient was initially admitted for starvation ketosis, as she reported poor oral intake for three days prior to admission."

### JSON

#### Example 1: single text document

In [5]:
input_json_data = {"text": sample_text}
response_json = invoke_realtime_endpoint(input_json_data, content_type="application/json", accept="application/json")
pd.DataFrame(response_json["predictions"])

,begin,end,ner_chunk,ner_label,ner_confidence,concept_code,resolution,score,billable_hcc,all_codes,concept_name_detailed,all_resolutions,all_score
0,39,67,gestational diabetes mellitus,DIABETES,0.96653336,O24.4,gestational diabetes mellitus,1.000000,"[, 0||0||0, 0||0||0, 0||0||0, ]","[O24.4, O24.41, O24.41, O24.41, O24.41]","[, gestational diabetes mellitus (disorder) [gestational diabetes mellitus in pregnancy], gestational diabetes [gestational diabetes mellitus in pregnancy], gdm - gestational diabetes mellitus [gestational diabetes mellitus in pregnancy], ]","[gestational diabetes mellitus, gestational diabetes mellitus (disorder), gestational diabetes, gdm - gestational diabetes mellitus, gestational diabetes mellitus in pregnancy]","[1.0000001192092896, 0.9637090563774109, 0.925167441368103, 0.9209029078483582, 0.9168061017990112]"
1,117,153,subsequent type two diabetes mellitus,PROBLEM,0.77357996,E13.9,secondary diabetes mellitus,0.849519,"[1||1||38, 1||1||38, 0||0||0, 1||1||38, 1||0||0]","[E13.9, E11.9, O24.11, E13.9, Z86.39]","[secondary diabetes mellitus [other specified diabetes mellitus without complications], diabetes mellitus type 2 [type 2 diabetes mellitus without complications], pre-existing type 2 diabetes mellitus [pre-existing type 2 diabetes mellitus, in pregnancy], secondary diabetes mellitus (disorder) [other specified diabetes mellitus without complications], history of diabetes mellitus type 2 [personal history of other endocrine, nutritional and metabolic disease]]","[secondary diabetes mellitus, diabetes mellitus type 2, pre-existing type 2 diabetes mellitus, secondary diabetes mellitus (disorder), history of diabetes mellitus type 2]","[0.8495187759399414, 0.8261895179748535, 0.8239017128944397, 0.8194190263748169, 0.8158331513404846]"
2,172,189,an acute hepatitis,PROBLEM,0.9745667,K72.0,acute hepatitis,0.900640,"[0||0||0, , 1||0||0, 1||0||0, 1||0||0]","[K72.0, B15, Z03.89, B17.9, Z03.89]","[acute hepatitis [acute and subacute hepatic failure], , acute hepatitis caused by infection suspected [encounter for observation for other suspected diseases and conditions ruled out], acute viral hepatitis [acute viral hepatitis, unspecified], acute hepatitis caused by infection suspected (situation) [encounter for observation for other suspected diseases and conditions ruled out]]","[acute hepatitis, acute hepatitis a, acute hepatitis caused by infection suspected, acute viral hepatitis, acute hepatitis caused by infection suspected (situation)]","[0.9006404876708984, 0.8742558360099792, 0.8741283416748047, 0.8730583190917969, 0.8706687688827515]"
3,196,202,obesity,OBESITY,0.7403,E66.9,obesity,1.000000,"[1||0||0, , 1||0||0, 1||0||0, 0||0||0]","[E66.9, E66, E66.9, E66.9, E66.8]","[obesity [obesity, unspecified], , alimentary obesity [obesity, unspecified], obesity (disorder) [obesity, unspecified], abdominal obesity [other obesity]]","[obesity, overweight and obesity, alimentary obesity, obesity (disorder), abdominal obesity]","[1.0, 0.8891977071762085, 0.8856316804885864, 0.8825730085372925, 0.8694376945495605]"
4,209,225,a body mass index,PROBLEM,0.845925,Z68,body mass index [bmi],0.870870,"[, 1||1||48, 1||0||0, 1||0||0, 1||0||0]","[Z68, Z68.41, E66.9, Z68.1, E66.9]","[, finding of body mass index [body mass index [bmi] 40.0-44.9, adult], observation of body mass index [obesity, unspecified], finding of body mass index (finding) [body mass index [bmi] 19.9 or less, adult], increased body mass index [obesity, unspecified]]","[body mass index [bmi], finding of body mass index, observation of body mass index, finding of body mass index (finding), increased body mass index]","[0.8708702325820923, 0.8615624904632568, 0.8383893966674805, 0.8280795812606812, 0.827699601650238]"
5,285,292,polyuria,SYMPTOM,0.9766,R35,polyuria,1.000000,"[, 0||0||0, , , ]","[R35, R35, R35.89, R35.8, R35.81]","[, polyuria (finding) [polyuria], , , ]","[polyuria, polyuria (finding), other polyuria, other polyuria, nocturnal polyuria]","[0

#### Example 2: array of text documents

In [6]:
input_json_data = {"text": docs}
response_json = invoke_realtime_endpoint(input_json_data, content_type="application/json", accept="application/json")
response_json

{'predictions': [[{'begin': 27,
    'end': 51,
    'ner_chunk': 'persistent abdominal pain',
    'ner_label': 'PROBLEM',
    'ner_confidence': '0.9549',
    'concept_code': 'K80.5',
    'resolution': 'recurrent abdominal pain',
    'score': 0.8763471245765686,
    'billable_hcc': ['0||0||0', '0||0||0', '1||0||0', '1||0||0', '1||0||0'],
    'all_codes': ['K80.5', 'K80.5', 'R10.9', 'R10.0', 'R10.9'],
    'concept_name_detailed': ['recurrent abdominal pain [calculus of bile duct without cholangitis or cholecystitis]',
     'recurrent abdominal pain (finding) [calculus of bile duct without cholangitis or cholecystitis]',
     'refractory abdominal pain [unspecified abdominal pain]',
     'recurrent acute abdominal pain [acute abdomen]',
     'nonspecific abdominal pain [unspecified abdominal pain]'],
    'all_resolutions': ['recurrent abdominal pain',
     'recurrent abdominal pain (finding)',
     'refractory abdominal pain',
     'recurrent acute abdominal pain',
     'nonspecific abdomi

### JSON Lines

In [7]:
def create_jsonl(records):
    if isinstance(records, str):
        records = [records]
    json_records = [{"text": text} for text in records]
    json_lines = "\n".join(json.dumps(record) for record in json_records)
    return json_lines

In [8]:
input_jsonl_data = create_jsonl(docs)
data = invoke_realtime_endpoint(input_jsonl_data, content_type="application/jsonlines" , accept="application/jsonlines" )
print(data)

{"predictions": [{"begin": 27, "end": 51, "ner_chunk": "persistent abdominal pain", "ner_label": "PROBLEM", "ner_confidence": "0.9549", "concept_code": "K80.5", "resolution": "recurrent abdominal pain", "score": 0.8763471245765686, "billable_hcc": ["0||0||0", "0||0||0", "1||0||0", "1||0||0", "1||0||0"], "all_codes": ["K80.5", "K80.5", "R10.9", "R10.0", "R10.9"], "concept_name_detailed": ["recurrent abdominal pain [calculus of bile duct without cholangitis or cholecystitis]", "recurrent abdominal pain (finding) [calculus of bile duct without cholangitis or cholecystitis]", "refractory abdominal pain [unspecified abdominal pain]", "recurrent acute abdominal pain [acute abdomen]", "nonspecific abdominal pain [unspecified abdominal pain]"], "all_resolutions": ["recurrent abdominal pain", "recurrent abdominal pain (finding)", "refractory abdominal pain", "recurrent acute abdominal pain", "nonspecific abdominal pain"], "all_score": [0.8763471245765686, 0.8688497543334961, 0.8286548256874084,